In [2]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("Sleep_health_and_lifestyle_dataset.csv")

In [3]:
df.columns = [c.strip() for c in df.columns]

for col in ["Gender", "Occupation", "BMI Category", "Blood Pressure", "Sleep Disorder"]:
    df[col] = df[col].astype(str).str.strip()


In [4]:
df["Sleep Disorder"] = df["Sleep Disorder"].replace({"nan": np.nan})
df["Sleep Disorder"] = df["Sleep Disorder"].fillna("None")

df["BMI Category"] = df["BMI Category"].replace({"Normal Weight": "Normal"})

In [5]:
def parse_bp(x):
    m = re.match(r"^\s*(\d{2,3})\s*/\s*(\d{2,3})\s*$", str(x))
    return (int(m.group(1)), int(m.group(2))) if m else (np.nan, np.nan)

bp = df["Blood Pressure"].apply(parse_bp)
df["Systolic_BP"] = bp.apply(lambda t: t[0])
df["Diastolic_BP"] = bp.apply(lambda t: t[1])

In [6]:
num_cols = ["Age","Sleep Duration","Quality of Sleep","Physical Activity Level",
            "Stress Level","Heart Rate","Daily Steps","Systolic_BP","Diastolic_BP"]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    

In [7]:
df["Age Group"] = pd.cut(df["Age"], [0,29,39,49,59,200], labels=["20s","30s","40s","50s","60+"])
df["Sleep Adequacy"] = pd.cut(df["Sleep Duration"], [0,6.99,7.99,24],
                              labels=["<7h (Low)","7-8h (Recommended)",">8h (High)"])
df["Steps Group"] = pd.cut(df["Daily Steps"], [0,4999,7499,9999,20000],
                           labels=["<5k","5k-7.5k","7.5k-10k","10k+"])
df["Stress Category"] = pd.cut(df["Stress Level"], [0,4,6,10], labels=["Low","Medium","High"])

df["Meets 7h Sleep"] = df["Sleep Duration"] >= 7
df["High Stress"] = df["Stress Level"] >= 7
df["Active (>=7.5k steps)"] = df["Daily Steps"] >= 7500

In [9]:
state_ranges = [
    (3, "Alabama"), (6, "Arizona"), (9, "California"),
    (12, "Colorado"), (15, "Florida"), (18, "Georgia"),
    (21, "Illinois"), (24, "Indiana"), (27, "Iowa"),
    (30, "Kansas"), (33, "Kentucky"), (36, "Louisiana"),
    (39, "Maryland"), (42, "Massachusetts"), (45, "Michigan"),
    (48, "Minnesota"), (51, "Missouri"), (54, "Nevada"),
    (57, "New Jersey"), (60, "New York"), (63, "North Carolina"),
    (66, "Ohio"), (69, "Oklahoma"), (72, "Oregon"),
    (75, "Pennsylvania"), (78, "South Carolina"), (81, "Tennessee"),
    (84, "Texas"), (87, "Utah"), (90, "Virginia"),
    (93, "Washington"), (96, "Wisconsin"), (999, "Wyoming")
]

def assign_region(pid):
    for max_id, state in state_ranges:
        if pid <= max_id:
            return state

df["Region"] = df["Person ID"].apply(assign_region)

In [10]:
df.to_csv("Sleep_health_and_lifestyle_dataset_cleaned.csv", index=False)